# NB2 · Preparing the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What this notebook does

In NB1 you reached the data. A raw table cannot be handed to a model directly; three more
pieces of work are needed.

First the data is cleaned: missing and impossible values are dealt with. Then it is split
into two groups; one teaches the model and the other tests it. Last, it is converted into
the form a model understands.

The order matters, and the third step coming last is not an accident. You will see why in
that step.


## Setup


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Code carried from the previous notebook

Paste the whole block collected at the end of the previous notebook into the cell
below. Do not delete the `#@cdss` marker on the first line; that block is collected
again at the end of this notebook and carried to the next one.

Running the block rebuilds everything you wrote in the earlier notebooks. Where it
reads data from the web, the cell may take a few seconds.


In [ ]:
#@cdss onceki_defter
# Paste the generated code below this line.


### Check · The carried code


In [ ]:
kit.check_defined('pd', 'np', 'cohort', 'DECISION_WINDOW_HOURS', 'TARGET_THRESHOLD_DAYS')


In [ ]:
kit.check_frame(cohort, name='cohort', required=['patient_id', 'target'], min_rows=30)


---

## Step 1 · Cleaning

Real hospital data does not arrive complete. Some measurements were never taken, some
were entered wrongly, some records are duplicated.

This step does three things. Entirely empty columns are dropped; a column holding no
information is nothing but a burden to the model. Duplicated rows are removed.
Physiologically impossible values are treated as missing.

The last item needs care. Rather than deleting a patient whose age was entered as 200, we
empty that value, because the rest of the patient's information is still usable. Which
value counts as impossible is a clinical decision and you write it into the prompt.


### Prompt 1

```
Write a piece of work that cleans the cohort table. Name it clean_data, let it clean
whatever table it is given and give the cleaned table back.

Have it do the following:
1. Drop columns that are entirely empty.
2. Drop rows that are exact duplicates.
3. Where age is below 0 or above 120, treat that value as missing. Do not delete the
   row, only empty that cell.
4. Drop rows where the target is empty; we do not know the right answer for them.

Have it print how many columns were dropped, how many duplicate rows were removed and
how many rows remain.

Then run it on cohort and keep the result under the name clean.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named clean_data that returns a table.
There must be a table named clean containing patient_id and target.
No empty value may remain in the target column of clean.
```


In [ ]:
#@cdss temizleme
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_function('clean_data', call_with=((cohort,), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(clean, name='clean', required=['patient_id', 'target'], min_rows=30)

print('\nEmpty values in target:', int(clean['target'].isna().sum()))


### Python note · Missing values and conditions

In the code you will see `NaN`. It means *unknown*, not zero. Zero is a measurement;
`NaN` is the absence of one. In clinical data the distinction is decisive: a patient whose
lactate was never measured is not the same as a patient whose lactate came back at zero.

Lines beginning with `if` are **conditions**: something is done when a given situation
holds. The age check in item three is written this way.

Expressions such as `df[df['age'] < 120]` are **filters**. The inner part produces true or
false for every row and the outer part keeps only the true ones. This is the most common
way of selecting rows from a table.

How missing values will be filled has not been decided yet. That work belongs to step
three, where the reason for the delay becomes clear.


In [ ]:
# How much is still missing in each column after cleaning.
missing = clean.isna().mean().sort_values(ascending=False)
print(missing.head(8).round(3).to_string())


---

## Step 2 · Training and test groups

A model learns from the data it is given. Testing it on the same data tells us nothing
about how useful it is; it measures what the model memorised. The data is therefore split
in two. The model learns from one group and is tested on the other, which it has never
seen.

How the split is made is critical. In the summary in NB1 you saw that the patient count
was lower than the row count: some patients have more than one stay. If rows are split at
random, one stay of a patient falls into the learning group and another into the test
group. The model recognises that patient and the test result comes out better than it is.

**The split must be made at patient level.** A patient belongs entirely to one group.
Generative AI tools do not do this of their own accord; you have to ask for it.


### Prompt 2

```
Split the clean table into two groups: the group the model will learn from and the group
it will be tested on.

Make the split at patient level. All records of the same patient must stay in one group;
no patient may appear in both. The patient identifier is in the patient_id column.

Let the test group be about 30 percent of the data. Use RANDOM_SEED for reproducibility.

Print the row count, patient count and rate of the target condition for both groups. Warn
me if the two rates differ markedly.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be two tables named train and test.
No patient_id may appear in both tables.
The target column must take both 0 and 1 in each table.
```


In [ ]:
#@cdss ayrim
# Paste the generated code below this line.


### Check 2


In [ ]:
kit.check_split(train, test, target='target', patient_id='patient_id')


### Python note · Why we split

A student who sees the exam questions in advance spoils what the exam measures. The same
holds for a model.

The test group must consist of patients the model has never seen. When two records of the
same patient are spread across both groups, the model recognises that patient's features
and scores well on the exam. That score is not repeated in the hospital.

The check cell tested this directly: it looked at the intersection of patient identifiers
in the two groups. A non-empty intersection means the split was made at row level and the
code has to be corrected.

The same problem arises with image and signal data. Two images or two signal segments
from the same patient must not be spread across the groups either.


---

## Step 3 · Converting into the form a model understands

A model works with numbers. Our table holds non-numeric information such as sex, type of
admission and insurance, which has to be converted. Missing values also have to be filled
and numeric columns brought onto comparable scales.

**How these conversions are performed is learned from the training group alone.** For
example, if a median age is used to fill missing ages, that median is computed from the
training group only and applied unchanged to the test group.

This rule is the most easily skipped point in the notebook. If the median is computed
across all the data, information about the test patients leaks into the learning process.
The result is leakage again; it raises no error, lifts the test result somewhat and looks
reasonable when the code is read.

That is why the third step comes after the split.


### Prompt 3

```
Convert the train and test tables into the form a model can use.

Do the following:
1. Separate the target column; call the training target y_train and the test target
   y_test.
2. Do not give identifier columns such as patient_id and stay_id to the model; they are
   numbers, not information about the patient.
3. Convert non-numeric columns into numbers.
4. Fill missing values.
5. Bring numeric columns onto comparable scales.

This matters a great deal: how the conversions in items three, four and five are
performed must be learned from the TRAINING group ONLY, then applied unchanged to the
test group. No information from the test group may enter the learning process. State in
a comment how you ensured this.

Print how many rows and columns each group now has.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be four objects named X_train, X_test, y_train and y_test.
X_train and X_test must have the same number of columns.
The row count of X_train must equal the length of y_train, and the row count of X_test
the length of y_test.
No empty value may remain in them.
```


In [ ]:
#@cdss hazirlama
# Paste the generated code below this line.


### Check 3


In [ ]:
kit.check_defined('X_train', 'X_test', 'y_train', 'y_test')


In [ ]:
import numpy as _np

print('Shape of X_train  :', _np.shape(X_train))
print('Shape of X_test   :', _np.shape(X_test))
print('Same column count :', _np.shape(X_train)[1] == _np.shape(X_test)[1])
print('Row counts match  :',
      _np.shape(X_train)[0] == len(y_train) and _np.shape(X_test)[0] == len(y_test))
print('Empty values left :', bool(_np.isnan(_np.asarray(X_train, dtype=float)).any()))


### Python note · Learning and applying

In the code you will see two operations named `fit` and `transform`. The difference
between them is the whole of this step.

`fit` **learns**: it computes the median, determines which categories exist, and derives
the mean and spread used for scaling. `transform` **applies**: it converts the table using
those learned values.

The correct use is this: on the training group `fit` and `transform` run together; on the
test group only `transform` runs. Calling `fit` on the test group produces leakage.

You may also see the word `Pipeline` in your code. It ties every conversion step into one
chain and ensures the whole chain is learned from the training group alone. It is the
preferred route because it makes the error structurally impossible.

Check whether your code calls `fit` on the test group. If it does, return it to the AI
tool and have it corrected.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB3.

The block is also saved as `cdss_nb2.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb2.py')


## What this notebook did

Three layers were added to the system: cleaning, a patient level split and conversion
into the form a model understands.

Two routes for leakage were closed. The first by splitting at patient level, the second
by learning the conversions from the training group alone. Both are errors that raise no
message, improve the result and are therefore hard to notice.

In NB3 a model is built on this data, taught and measured.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
